# 02b - Experiment Tracking: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 02, you trained text classifiers on inspection comments using TF-IDF.
Now you'll train the **same three classifiers** on structured tabular data —
predicting whether a service order is an **Overhaul** or **Preventive**.

### What you'll do:
1. Set up MLflow tracking (same as Track A)
2. Prepare tabular features (one-hot encoding + scaling)
3. Train Logistic Regression, Random Forest, and Gradient Boosting
4. Compare runs — see how the same models behave differently on tabular vs text data

### Key difference from Track A:
- **Track A** features: TF-IDF vectors from text (sparse, 5000 dimensions)
- **Track B** features: One-hot encoded categoricals + scaled numerics (dense, ~70 dimensions)
- **Same MLflow interface:** tracking, metrics, artifacts — all identical

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Configure MLflow Tracking

We use a **separate experiment name** from Track A so runs don't mix.

In [ ]:
import mlflow

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
tracking_uri = workspace.mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {tracking_uri}")

EXPERIMENT_NAME = "contoso-repair-classifier"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")

## Step 2: Load and Prepare Data

We use the cleaned dataset (v2) that we registered in Notebook 01b.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_b_tabular"))

import pandas as pd
from preprocess_os import load_and_clean_os, prepare_train_test

df = load_and_clean_os("../../data/service_orders_dataset.csv")
print(f"Dataset: {len(df):,} rows")
print(f"Label distribution: {df['label'].value_counts().rename({0: 'Preventive', 1: 'Overhaul'}).to_dict()}")

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, train_df, test_df = prepare_train_test(
    df, test_size=0.2, max_categories=20
)

print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]:,} features")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"Train label dist: {pd.Series(y_train).value_counts().rename({0: 'Preventive', 1: 'Overhaul'}).to_dict()}")
print(f"\nNote: Compare with Track A's 5,000 TF-IDF features — tabular data is much more compact.")

## Step 3: Train Multiple Models with MLflow Tracking

Same three classifiers as Track A. The `train_and_log` function is nearly
identical — the only difference is the data lineage tags and label names.

### Run 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np
import joblib

def train_and_log(model, model_name, X_train, X_test, y_train, y_test, preprocessor, data_version="2"):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_param("data_asset", "service-orders")
        mlflow.log_param("data_version", data_version)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_train", X_train.shape[0])
        mlflow.log_param("n_test", X_test.shape[0])
        mlflow.log_param("n_features", X_train.shape[1])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
        }
        mlflow.log_metrics(metrics)

        report = classification_report(y_test, y_pred, target_names=["Preventive", "Overhaul"])
        report_path = f"{model_name}_report.txt"
        with open(report_path, "w") as f:
            f.write(report)
        mlflow.log_artifact(report_path)

        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, display_labels=["Preventive", "Overhaul"], ax=ax, cmap="Oranges"
        )
        ax.set_title(f"{model_name} - Confusion Matrix")
        cm_path = f"{model_name}_confusion_matrix.png"
        fig.savefig(cm_path, dpi=100, bbox_inches="tight")
        mlflow.log_artifact(cm_path)
        plt.show()

        mlflow.sklearn.log_model(model, artifact_path="model")

        preprocessor_path = "preprocessor.joblib"
        joblib.dump(preprocessor, preprocessor_path)
        mlflow.log_artifact(preprocessor_path, artifact_path="model")

        os.remove(report_path)
        os.remove(cm_path)
        os.remove(preprocessor_path)

        print(f"\n{model_name} results:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        print(f"  Run ID: {run.info.run_id}")

        return run.info.run_id, metrics

In [ ]:
lr_run_id, lr_metrics = train_and_log(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=1.0),
    "LogisticRegression",
    X_train, X_test, y_train, y_test,
    preprocessor=preprocessor,
)

### Run 2: Random Forest

In [ ]:
rf_run_id, rf_metrics = train_and_log(
    RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
    "RandomForest",
    X_train, X_test, y_train, y_test,
    preprocessor=preprocessor,
)

### Run 3: Gradient Boosting

In [ ]:
gb_run_id, gb_metrics = train_and_log(
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42),
    "GradientBoosting",
    X_train, X_test, y_train, y_test,
    preprocessor=preprocessor,
)

### Run 4: Logistic Regression with Different Regularization

In [ ]:
lr2_run_id, lr2_metrics = train_and_log(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=0.1),
    "LogisticRegression_C0.1",
    X_train, X_test, y_train, y_test,
    preprocessor=preprocessor,
)

## Step 4: Compare Runs

In [ ]:
all_results = {
    "LogisticRegression": lr_metrics,
    "RandomForest": rf_metrics,
    "GradientBoosting": gb_metrics,
    "LogisticRegression_C0.1": lr2_metrics,
}

results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values("f1", ascending=False)
print("\nModel Comparison (sorted by F1):")
print(results_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, metric in enumerate(["accuracy", "precision", "recall", "f1"]):
    results_df[metric].plot(kind="barh", ax=axes[i], color=["#e377c2", "#17becf", "#bcbd22", "#9467bd"])
    axes[i].set_title(metric.capitalize())
    axes[i].set_xlim(0, 1)
plt.suptitle("Contoso Repair Classifier - Model Comparison (Tabular)", fontsize=14)
plt.tight_layout()
plt.show()

### Query Runs via MLflow API

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=20,
)

finished_runs = [r for r in runs if r.info.status == "FINISHED" and "f1" in r.data.metrics]
finished_runs.sort(key=lambda r: r.data.metrics.get("f1", 0), reverse=True)

print("Top runs by F1 score:")
for run in finished_runs[:5]:
    print(f"  {run.data.params.get('model_type', 'N/A'):30s} | "
          f"F1: {run.data.metrics.get('f1', 0):.4f} | "
          f"Run: {run.info.run_id[:8]}")

## Go to Azure ML Studio Now

Navigate to **Jobs > contoso-repair-classifier** and explore:

1. **Run list** — See all 4 runs with their metrics
2. **Select 2+ runs > Compare** — Use the comparison view
3. Compare with **contoso-lead-classifier** (Track A) — same models, different data

Notice how the MLflow UI is identical whether you're tracking text or tabular experiments.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Separate experiments** | Track A and B have independent experiment namespaces |
| **Same tracking API** | `mlflow.log_param`, `log_metrics`, `log_artifact` — identical |
| **Data lineage** | Each run records `service-orders` v2 (vs `classified-inspections` v2 in Track A) |
| **Feature engineering** | One-hot encoding + scaling instead of TF-IDF — same MLflow tracking |

---

**Next:** [03b - Model Registration (Tabular)](./03b_model_registration_tabular.ipynb)